In [15]:
import re, json, yaml

patterns_map = dict(
    # Identifiers
    attemptId=r'attempt with (?:the )?id',
    categoryId=r'category with id',
    chapterId=r'chapter with id',
    choiceId=r'choice with id',
    commentId=r'comment with id',
    courseId=r'course with (?:the )?id|old course with id|new course with id|curso:',
    discussionId=r'discussion(?: with id)?',
    enrolmentId=r'enrolment method .*? with id',
    eventId=r'event .*? with id',
    evidenceId=r'evidence with id',
    fieldId=r'field with id',
    forumId=r'forum(?: with id)?',
    glossaryEntryId=r'glossary entry with id',
    gradeId=r'grade with id',
    gradeItemId=r'grade item with id',
    groupId=r'group with id',
    groupingId=r'grouping with id',
    h5pId=r'H5P with the id',
    itemId="Item(?: created)? with ID|item type ''.*?'' with id",
    moduleId=r'course module(?: with)? id',
    noteId=r'note with id',
    optionId=r'option with id',
    overrideId=r'override with id',
    pageId=r'page with (?:the )?id',
    postId=r'(?:forum )?post with id',
    questionId=r'question with id',
    questionCategoryId=r'question category with id',
    recordId=r'data record with id',
    roleId=r'role with id',
    ruleId=r'rule with id',
    scoId=r'sco with id',
    sectionId=r'section number|section with id',
    stepId=r'\(id',
    submissionId=r'submission(?: with id(?: of)?)?',
    subscriptionId=r'subscription(?: with id)?',
    tagId=r'tag with id',
    tourId=r'tour with id',
    userCompetencyId=r'user(?: course)? competency with id',
    userId=r'user with (?:the )?id|user',
    fileCount=r'uploaded',
    stepIndex=r'step index',
    wordCount=r'submission with',
    ratingValue=r'with(?= ''INTEGER'' rating)',
    scormValue=r'value of'
)

# Infiere campos dentro del texto de una descripción; Por ejemplo:
# `The user with id '20' subscribed the user with id '20' to the discussion  with id '214' in the forum with the course module id '1169'.`
# Campos en orden de aparición: userId, targetUserId, discussionId, moduleId.
#
# De manera que, extrayendo los números enteros en orden y emparejándolos con sus campos se tiene:
# Campos:  userId targetUserId discussionId moduleId
# Valores: 20     20           214          1169
#
def infer_fields(description):
    matches = []
    for field, pattern in patterns_map.items():
        for match in re.finditer(pattern, description, re.IGNORECASE):
            matches.append((match.start(), field))

    # Las tuplas son del tipo: (posición, identificador),
    # por lo que al ordenarlas por posición, valga la redundancia,
    # los campos se guardan en el orden de aparición en la descripción
    matches.sort(key=lambda x: x[0])
    fields = [match[1] for match in matches]

    # Si el campo de usuario aparece por segunda vez,
    # el usuario es el objetivo de la acción (targetUserId)
    user_count = 0
    final_fields = []
    for field in fields:
        if field == "userId":
            user_count += 1
            final_fields.append("targetUserId" if user_count > 1 else "userId")
        else:
            final_fields.append(field)

    return final_fields

def transform(node):
    if isinstance(node, dict):
        return {k: transform(v) for k, v in node.items()}
    elif isinstance(node, list):
        if len(node) == 1:
            return infer_fields(node[0])
        else:
            return [infer_fields(item) for item in node]
    return node

with open('../components.json') as f:
    data = json.load(f)

mappings = transform(data)

# Impresión de la configuración para el backend.
#
# flow_style=None
# Se evita el flow_style=True porque hace YAML más verboso, utilizando llaves y comas, asemejándose a JSON;
# Tampoco se descarta por completo (flow_style=False) porque se prefieren las listas inline [...].
#
# sort_keys=True
# No es estrictamente necesario, pero se mantienen los componentes y eventos ordenados
#
# allow_unicode=True
# Tampoco es estrictamente necesario ya que se trabaja sobre strings en language=en,
# además, independientemente del idioma, los identificadores mantienen su posición,
# pero se conserva la compatibilidad con Unicode para evitar imprevistos.
#
print(yaml.dump(mappings, default_flow_style=None, sort_keys=True, allow_unicode=True))

Activity report:
  Activity report viewed: [userId, courseId]
  Outline report viewed: [userId, targetUserId, courseId]
Assignment:
  A submission has been submitted.: [userId, submissionId, wordCount, moduleId]
  All the submissions are being downloaded.: [userId, submissionId, moduleId]
  An extension has been granted.: [userId, targetUserId, moduleId]
  Assignment override created: [userId, overrideId, moduleId, targetUserId]
  Course module instance list viewed: [userId, courseId]
  Course module viewed: [userId, moduleId]
  Grading form viewed: [userId, targetUserId, moduleId]
  Grading table viewed: [userId, moduleId]
  Submission confirmation form viewed.: [userId, submissionId, moduleId]
  Submission form viewed.:
  - [userId, submissionId, moduleId]
  - [userId, submissionId, targetUserId, moduleId]
  Submission viewed.: [userId, submissionId, targetUserId, moduleId]
  The state of the workflow has been updated.: [userId, targetUserId, moduleId]
  The status of the submission 